In [1]:
import os
import shutil
import re

# Folder setup
log_folder = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs"
run_log_path = os.path.join(log_folder, "run_log.txt")
success_folder = os.path.join(log_folder, "successful")
fail_folder = os.path.join(log_folder, "failed")

os.makedirs(success_folder, exist_ok=True)
os.makedirs(fail_folder, exist_ok=True)

# Parse run_log.txt
success_hashes = []
fail_hashes = []

with open(run_log_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    match = re.match(r"==== Processing (.+\.mod) ====", line)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)

# Move .log files based on success/failure
def move_log_files(hashes, destination):
    for hash_id in hashes:
        log_file = f"{hash_id}.log"
        src = os.path.join(log_folder, log_file)
        dst = os.path.join(destination, log_file)
        if os.path.exists(src):
            shutil.copy2(src, dst)

move_log_files(success_hashes, success_folder)
move_log_files(fail_hashes, fail_folder)

print(f"✅ Moved {len(success_hashes)} successful logs to {success_folder}")
print(f"❌ Moved {len(fail_hashes)} failed logs to {fail_folder}")


✅ Moved 426 successful logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/successful
❌ Moved 128 failed logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/failed


In [3]:
log_files = set(f.replace(".log", "") for f in os.listdir(log_folder) if f.endswith(".log"))
mentioned_hashes = set(success_hashes + fail_hashes)
unmatched_logs = log_files - mentioned_hashes
print("Not matched in run_log.txt:", unmatched_logs)

Not matched in run_log.txt: set()
